# Used-Car Data Cleaning and Preprocessing

## Objective

The objective of this project is to clean a messy used-car dataset and convert it into a machine-learning-ready numerical dataset.

The workflow includes:

1. Loading the raw dataset
2. Setting `car_id` as the index
3. Cleaning categorical columns
4. Converting string-formatted numerical columns
5. Removing duplicate rows
6. Handling missing values
7. Removing invalid/outlier records
8. Encoding categorical variables
9. Returning a summary of the cleaned dataset

## Problem Statement

The dataset contains several data-quality problems:

- Extra spaces in categorical values
- Inconsistent capitalization
- String-formatted numerical values
- Missing values
- Duplicate records
- Invalid prices
- Invalid seat values
- Categorical columns that must be converted into numerical form

The goal is to produce a clean dataset suitable for machine learning.

In [1]:
import pandas as pd
from io import StringIO

In [2]:
# ## Load the Raw Dataset

# The sample data is provided directly as a CSV string.

# We use `StringIO` to allow Pandas to treat the string like a CSV file

RAW_CSV = """car_id,brand,transmission,fuel_type,km_driven,mileage,seats,price
1,Maruti,Manual,petrol,"45,000 KM",21.4 KMPL,5.0,350000
2, Honda ,Automatic,DIESEL,"30,000",19.1 KMPL,,280000
3,Maruti,Manual,Petrol,"45,000 KM",21.4 KMPL,5.0,350000
4,BMW,Automatic,diesel,"12,000 KM",14.3 KMPL,5.0,1200000
5,Ferrari,Manual,Petrol,"1,200 KM",,2.0,9999999
6, Honda ,manual,petrol,"28,000 KMS",18.9 KMPL,5.0,
7,Maruti,Manual,Petrol,"45,000 KM",21.4 KMPL,5.0,350000
8,Toyota,Automatic,Diesel,"55,000 KM",16.7 KMPL,7.0,650000
9,BMW,Automatic,Diesel,"11,000 KM",14.3 KMPL,5.0,800
10,Hyundai,Manual,CNG,"20,000 KM",24.1 KMPL,0.0,420000
"""

In [3]:
df = pd.read_csv(StringIO(RAW_CSV))

df

,car_id,brand,transmission,fuel_type,km_driven,mileage,seats,price
0,1,Maruti,Manual,petrol,"45,000 KM",21.4 KMPL,5.0,350000.0
1,2,Honda,Automatic,DIESEL,"30,000",19.1 KMPL,NaN,280000.0
2,3,Maruti,Manual,Petrol,"45,000 KM",21.4 KMPL,5.0,350000.0
3,4,BMW,Automatic,diesel,"12,000 KM",14.3 KMPL,5.0,1200000.0
4,5,Ferrari,Manual,Petrol,"1,200 KM",NaN,2.0,9999999.0
5,6,Honda,manual,petrol,"28,000 KMS",18.9 KMPL,5.0,NaN
6,7,Maruti,Manual,Petrol,"45,000 KM",21.4 KMPL,5.0,350000.0
7,8,Toyota,Automatic,Diesel,"55,000 KM",16.7 KMPL,7.0,650000.0
8,9,BMW,Automatic,Diesel,"11,000 KM",14.3 KMPL,5.0,800.0
9,10,Hyundai,Manual,CNG,"20,000 KM",24.1 KMPL,0.0,420000.0


In [4]:
# ## Inspect the Raw Dataset

# Before cleaning the data, we inspect its structure and missing values.
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (10, 8)

Columns:
['car_id', 'brand', 'transmission', 'fuel_type', 'km_driven', 'mileage', 'seats', 'price']

Data Types:
car_id            int64
brand            object
transmission     object
fuel_type        object
km_driven        object
mileage          object
seats           float64
price           float64
dtype: object

Missing Values:
car_id          0
brand           0
transmission    0
fuel_type       0
km_driven       0
mileage         1
seats           1
price           1
dtype: int64


In [5]:
# ## Set `car_id` as the Index

# `car_id` is an identifier and should not be used as a machine-learning feature.

# It is also important to set it as the index before duplicate detection so that the identifier does not prevent duplicate records from being detected.
df = df.set_index("car_id")

df.head()

,brand,transmission,fuel_type,km_driven,mileage,seats,price
car_id,,,,,,,
1,Maruti,Manual,petrol,"45,000 KM",21.4 KMPL,5.0,350000.0
2,Honda,Automatic,DIESEL,"30,000",19.1 KMPL,NaN,280000.0
3,Maruti,Manual,Petrol,"45,000 KM",21.4 KMPL,5.0,350000.0
4,BMW,Automatic,diesel,"12,000 KM",14.3 KMPL,5.0,1200000.0
5,Ferrari,Manual,Petrol,"1,200 KM",NaN,2.0,9999999.0


In [6]:
# ## Step 1 — Clean Categorical Columns

# The categorical columns contain extra spaces and inconsistent capitalization.

# We will use:

# - `str.strip()` to remove unnecessary spaces.
# - `str.title()` to standardize capitalization.

# Columns:

# - brand
# - transmission
# - fuel_type

text_cols = [
    "brand",
    "transmission",
    "fuel_type"
]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

df[text_cols]

,brand,transmission,fuel_type
car_id,,,
1,Maruti,Manual,Petrol
2,Honda,Automatic,Diesel
3,Maruti,Manual,Petrol
4,Bmw,Automatic,Diesel
5,Ferrari,Manual,Petrol
6,Honda,Manual,Petrol
7,Maruti,Manual,Petrol
8,Toyota,Automatic,Diesel
9,Bmw,Automatic,Diesel


In [7]:
# ## Step 2 — Replace Known Typos

# After normalizing the text, known spelling mistakes can be replaced.

# The provided sample does not contain an additional explicit typo, but this step is included to follow the required preprocessing workflow.

# Example typo correction structure.
# No explicit typo is present in the supplied sample.

typo_fixes = {
    # "Marruti": "Maruti",
    # "Hondaa": "Honda"
}

for col, replacements in {
    "brand": typo_fixes
}.items():
    df[col] = df[col].replace(replacements)

In [8]:
# ## Step 3 — Convert `km_driven`

# The `km_driven` column contains values such as:

# - `45,000 KM`
# - `30,000`
# - `28,000 KMS`

# These must become numerical values.

# We remove:

# - commas
# - `KM`
# - `KMS`

# Then use `pd.to_numeric(errors="coerce")`.
df["km_driven"] = (
    df["km_driven"]
    .str.replace(",", "", regex=False)
    .str.replace("KMS", "", regex=False)
    .str.replace("KM", "", regex=False)
    .str.strip()
)

df["km_driven"] = pd.to_numeric(
    df["km_driven"],
    errors="coerce"
)

df["km_driven"]


car_id
1     45000
2     30000
3     45000
4     12000
5      1200
6     28000
7     45000
8     55000
9     11000
10    20000
Name: km_driven, dtype: int64

## Step 4 — Convert `mileage`

The `mileage` column contains values such as:

`21.4 KMPL`

We remove the `KMPL` suffix and convert the result into numeric values.

In [9]:
df["mileage"] = (
    df["mileage"]
    .str.replace("KMPL", "", regex=False)
    .str.strip()
)

df["mileage"] = pd.to_numeric(
    df["mileage"],
    errors="coerce"
)

df["mileage"]

car_id
1     21.4
2     19.1
3     21.4
4     14.3
5      NaN
6     18.9
7     21.4
8     16.7
9     14.3
10    24.1
Name: mileage, dtype: float64

## Step 5 — First Duplicate Removal

After normalizing text and converting numerical values, duplicate records can now be detected correctly.

We use:

`drop_duplicates()`

The expected duplicate records correspond to car IDs:

- 3
- 7

Both duplicate the information of car ID 1.

In [10]:
before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)
print("Rows removed:", before_duplicates - after_duplicates)

Rows before duplicate removal: 10
Rows after duplicate removal: 8
Rows removed: 2


## Step 6 — Handle Missing Values

The missing-value strategy depends on the column.

### Rules

- Missing values in columns with less than 5% missing → drop the affected rows.
- Continuous numerical columns → median imputation.
- Integer/categorical columns → mode imputation.

The most important missing value in this sample is the missing `price`, which must be removed because `price` is the target and cannot be safely imputed for this task.

In [11]:
print(df.isnull().sum())

brand           0
transmission    0
fuel_type       0
km_driven       0
mileage         1
seats           1
price           1
dtype: int64


## Handle Missing Price

`price` is the target variable.

The sample contains one missing price value. Since this is below the 5% threshold specified by the problem, the corresponding row is removed.

In [12]:
df = df.dropna(subset=["price"])

print("Shape after removing missing price :", df.shape)

Shape after removing missing price : (7, 7)


## Impute Missing Mileage

`mileage` is a continuous numerical feature.

Therefore, the missing value is filled using the median.

In [13]:
mileage_median = df["mileage"].median()

df["mileage"] = df["mileage"].fillna(
    mileage_median
)

print("Mileage median:", mileage_median)

Mileage median: 17.9


## Impute Missing Seats

`seats` represents a discrete count.

According to the given rule, integer/discrete columns use mode imputation.

In [15]:
seats_mode = df["seats"].mode()[0]

df["seats"] = df["seats"].fillna(
    seats_mode
)

print("Seats mode:", seats_mode)

Seats mode: 5.0


## Step 7 — Remove Duplicates Again

After imputation, rows that were previously different could potentially become identical.

Therefore, the assignment explicitly requires a second:

`drop_duplicates()`

In [16]:
df = df.drop_duplicates()

print("Shape after second duplicate removal:", df.shape)

Shape after second duplicate removal: (7, 7)


## Step 8 — Remove Invalid Price Values

The problem specifies that rows with:

`price <= 10000`

must be removed.

The dataset contains:

`car_id = 9`

with:

`price = 800`

Therefore, this record is removed.

In [17]:
df = df[df["price"] > 10000]

print("Shape after price filtering:", df.shape)

Shape after price filtering: (6, 7)


## Step 9 — Remove Invalid Seat Values

A vehicle cannot have zero seats in this dataset.

The problem specifically says to remove rows where:

`seats == 0`

This removes car ID 10.

In [18]:
df = df[df["seats"] != 0]

print("Shape after seat filtering:", df.shape)

Shape after seat filtering: (5, 7)


## Step 10 — Encode Transmission

The problem specifically requires:

Manual → 0
Automatic → 1

In [19]:
df["transmission"] = df["transmission"].map({
    "Manual": 0,
    "Automatic": 1
})

df["transmission"]

car_id
1    0
2    1
4    1
5    0
8    1
Name: transmission, dtype: int64

## Step 11 — One-Hot Encode Fuel Type

The surviving records contain:

- Diesel
- Petrol

CNG was present in the original data but its record was removed because it had `seats = 0`.

Using `drop_first=True` makes Diesel the reference category and creates:

`fuel_type_Petrol`

In [20]:
df = pd.get_dummies(
    df,
    columns=["fuel_type"],
    drop_first=True,
    dtype=int
)

df

,brand,transmission,km_driven,mileage,seats,price,fuel_type_Petrol
car_id,,,,,,,
1,Maruti,0,45000,21.4,5.0,350000.0,1
2,Honda,1,30000,19.1,5.0,280000.0,0
4,Bmw,1,12000,14.3,5.0,1200000.0,0
5,Ferrari,0,1200,17.9,2.0,9999999.0,1
8,Toyota,1,55000,16.7,7.0,650000.0,0


In [21]:
# ## Step 12 — Group Rare Brands

# Brands appearing fewer than 2 times are grouped into:

# `Other`

# In the final surviving dataset, every brand appears only once.

# Therefore all five brands become `Other`.

brand_counts = df["brand"].value_counts()

rare_brands = brand_counts[
    brand_counts < 2
].index

df["brand"] = df["brand"].replace(
    rare_brands,
    "Other"
)

df["brand"]

car_id
1    Other
2    Other
4    Other
5    Other
8    Other
Name: brand, dtype: object

## Step 13 — One-Hot Encode Brand

After grouping rare brands, all remaining brands belong to the `Other` category.

Since there is only one category, `drop_first=True` produces no additional brand dummy column.

In [22]:
df = pd.get_dummies(
    df,
    columns=["brand"],
    drop_first=True,
    dtype=int
)

df

,transmission,km_driven,mileage,seats,price,fuel_type_Petrol
car_id,,,,,,
1,0,45000,21.4,5.0,350000.0,1
2,1,30000,19.1,5.0,280000.0,0
4,1,12000,14.3,5.0,1200000.0,0
5,0,1200,17.9,2.0,9999999.0,1
8,1,55000,16.7,7.0,650000.0,0


## Final Cleaned Dataset

The final dataset should contain:

- transmission
- km_driven
- mileage
- seats
- price
- fuel_type_Petrol

Expected shape:

`(5, 6)`

In [23]:
print("Final DataFrame:")
display(df)

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Final DataFrame:


,transmission,km_driven,mileage,seats,price,fuel_type_Petrol
car_id,,,,,,
1,0,45000,21.4,5.0,350000.0,1
2,1,30000,19.1,5.0,280000.0,0
4,1,12000,14.3,5.0,1200000.0,0
5,0,1200,17.9,2.0,9999999.0,1
8,1,55000,16.7,7.0,650000.0,0



Final Shape:
(5, 6)

Final Columns:
['transmission', 'km_driven', 'mileage', 'seats', 'price', 'fuel_type_Petrol']

Missing Values:
transmission        0
km_driven           0
mileage             0
seats               0
price               0
fuel_type_Petrol    0
dtype: int64

Duplicate Rows:
0


## Final Summary

The messy used-car dataset has been successfully transformed into a clean numerical dataset.

### Cleaning operations performed

1. Normalized categorical text.
2. Converted string-formatted numerical values.
3. Removed duplicate records.
4. Removed the row with missing target price.
5. Imputed missing mileage using the median.
6. Imputed missing seats using the mode.
7. Removed duplicate records again.
8. Removed invalid price values.
9. Removed vehicles with zero seats.
10. Encoded transmission numerically.
11. One-hot encoded fuel type.
12. Grouped rare brands into `Other`.
13. One-hot encoded the brand column.

The final dataset contains 5 rows, 6 columns, no missing values, and no duplicate rows.